# 🫀 퀘스트 46 · Q7-B′ — **전수 재채점** (DEV+TEST 합쳐 55개체)

| | **MedKOS / `notebooks/quest46_q7bp_svdb_full.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0052`(Q7-B) · `ailab-2026-0049`(Q7-A) |
| 규약 | `SCORING_RULES.md` **R4 · R11-c · R12 · R15 · R16** |

## 왜 별도 노트북인가

**Q7-B 노트북은 얼려 둔다.** 관문이 이미 발화한 노트북을 고치는 건 사후해석이 스며드는
전형적인 경로다. 이 실험은 **다른 질문**이고 **자기 사전등록**을 갖는다.

- Q7-B 가 물은 것: "이 코호트가 판정을 지탱하는가" → TEST 반쪽에서 관문 5개 통과
- Q7-B′ 가 묻는 것: **"전수에서도 유지되는가, 그리고 반쪽 격차는 우연인가"**

## 학습 0회

`data/q7b_svdb_probs_s5.npz` 의 예측을 그대로 읽는다. 모델을 다시 돌리지 않는다.

## 왜 전수 채점이 정당한가 (그리고 어디까지만 정당한가)

`GMIN=2` 는 Q7-B 가 **DEV 에서** 골랐다. 그 DEV 를 다시 채점에 넣으면 원칙적으로
선택 데이터를 재사용하는 것이다. 그런데 **그 선택이 소비한 정보가 사실상 0** 이다 —
DEV 스윕에서 **11개 GMIN 값이 전부 통과**했고 규칙(「가장 작은 통과 GMIN」)이 격자의
하한을 그대로 돌려줬다. 고를 게 없었으므로 전수 채점이 사후선택이 되지 않는다.

**그래도 역할은 갈라 둔다:**

| | |
|---|---|
| **사전등록 관문 판정** | Q7-B 의 **TEST 전용** 값이 최종이다. 여기서 바꾸지 않는다 |
| **Q2 를 위한 최선 추정** | 여기 **전수** 값을 쓴다(개체가 2배라 CI 가 좁다) |

## ID 매핑

Q7-B 에서 `_svdb_load` 의 조용한 fallback 때문에 레코드 번호가 어긋났다(→ **R16**).
`【Q7B-M】` 이 결정론적으로 복원해 `data/svdb_id_map_q7b.json` 에 저장했다.
이 노트북은 **그 매핑을 적용한 뒤** 시작하고, 적용 결과를 다시 검증한다.

## 사전등록

| 관문 | 내용 | 문턱 |
|---|---|---|
| **P1** | 전수 채점 개체 수 | **≥ 40** |
| **P2** | **최대** 부트 SE (R12 — 근사 금지) | **≤ 0.10** |
| **P3** | 지배 지분 (R11-3) | **≤ 50%** |
| **P4** | 매크로 AUROC 의 레코드 부트 95% CI 폭 | **≤ 0.10** |
| **P5** | **반쪽 격차가 우연으로 설명되나** — 순열검정 | **p ≥ 0.05** |

**P5 가 이 실험의 핵심이다.** Q7-B 에서 DEV 0.8382 vs TEST 0.9256 (격차 0.0874) 였고,
층화가 실제로는 작동하지 않았으므로(라벨이 어긋난 채 갈랐다) 그 분할은 **임의**였다.
임의 분할이라면 55개체를 무작위로 27/28 로 가르는 순열분포와 비교할 수 있다 —
관측 격차가 그 분포 안에 흔하면 **우연**이고, 꼬리에 있으면 **구조적**이다.

⚠️ **해석 비대칭**: p 는 성능이 아니라 **산포**에 대한 진술이다. 코호트가 균질할수록
같은 격차가 더 희귀해진다. 그래서 **지지 = 「구조적 차이의 증거가 없다」** 이지
「차이가 없다」가 아니고, **기각 = 「두 반쪽이 교환가능하지 않다」** 이지 「코호트가
나쁘다」가 아니다. 분할이 사실상 임의였는데도 기각이 나오면 그건 **설명이 필요한
사건**이다(예: 채점 가능 여부가 분할과 상관).

⚠️ 여전히 지키는 것: **전이 낙폭 인용 금지**(SVDB 128→360Hz 대역 교란 · Q7-C 몫) ·
**실험22-A·§6.5 와 직접 비교 불가**(WST 를 DS1 에서만 fit) · **PR 계열은 서술용**
(군·개체 비교는 AUROC 로만 — R15-d).


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

class AssetError(RuntimeError): pass
print("CELL 0 ✅ decide 준비")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0    = 20260803
GMIN     = 2         # ★ Q7-B 가 DEV 에서 고른 값. 여기서 다시 고르지 않는다
SE_MAX   = 0.10
N_MIN    = 40        # 전수 하한 (반쪽 20 의 두 배)
DOM_MAX  = 0.50
CIW_MAX  = 0.10
NB_BOOT  = 400
NB_MACRO = 4000
NB_PERM  = 20000     # P5 순열검정
IDX_S, IDX_V = 1, 2

CONFIG = dict(
    exp="quest46_q7bp_svdb_full", quest="ailab-2026-0046", step="svdb-full-rescore",
    parent_exp=["quest46_q7b_svdb_se", "ailab-2026-0052"],
    purpose=("Q7-B 는 TEST 반쪽(28개체)에서 관문 5개를 통과했다. 전수(DEV+TEST ≈55개체)"
             "에서도 유지되는지, 그리고 반쪽 격차(0.8382 vs 0.9256)가 우연인지 본다"),
    dataset="SVDB 전수 · Q7-B 예측 캐시 재사용 (학습 0회)",
    change_one_thing="채점 대상만 TEST 반쪽 → 전수. 모델·확률·GMIN 전부 그대로",
    gmin=GMIN,
    gmin_provenance=("Q7-B 가 DEV 에서 고른 값. DEV 스윕에서 11개 GMIN 이 **전부 통과**해 "
                     "규칙(가장 작은 통과 GMIN)이 격자 하한을 돌려줬다 — 선택이 소비한 "
                     "정보가 사실상 0 이므로 전수 채점이 사후선택이 되지 않는다"),
    role_separation=("사전등록 관문 판정은 Q7-B 의 **TEST 전용** 값이 최종이다. 여기 전수 "
                     "값은 **Q2 를 위한 최선 추정**으로 쓴다 — 두 역할을 섞지 않는다"),
    thresholds=dict(n_min=N_MIN, se_max=SE_MAX, dom_max=DOM_MAX, ciw_max=CIW_MAX, perm_p=0.05),
    predictions={
        "P1": f"전수 채점 개체 ≥ {N_MIN}",
        "P2": f"최대 부트 SE ≤ {SE_MAX}",
        "P3": f"지배 지분 ≤ {DOM_MAX:.0%}",
        "P4": f"매크로 AUROC 레코드 부트 95% CI 폭 ≤ {CIW_MAX}",
        "P5": "반쪽 격차가 순열분포 안에서 흔하다 (p ≥ 0.05) — 우연으로 설명됨"},
    caveat=("전이 낙폭 인용 금지(128→360Hz 대역 교란 · Q7-C 몫). 실험22-A·§6.5 와 직접 "
            "비교 불가(WST DS1-only fit). PR 계열은 서술용 — 군·개체 비교는 AUROC 로만(R15-d). "
            "`_medref`·BN 은 평가 입력을 쓰는 transductive 요소이고 관문은 raw 로 매긴다"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7bp_svdb_full", CONFIG, project=PROJECT)
run.log(f"설정 ✅ GMIN={GMIN}(Q7-B 에서 승계) · 전수 하한 {N_MIN} · 학습 0회")


In [ ]:
# CELL 2 — 【G0】 캐시 로드 + **ID 매핑 적용·재검증**
PROB = os.path.join(PROJECT, "data", "q7b_svdb_probs_s5.npz")
MAPJ = os.path.join(PROJECT, "data", "svdb_id_map_q7b.json")
CNTJ = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
for p, why in ((PROB, "Q7-B 예측 캐시"), (CNTJ, "Q7-A 주석 카운트")):
    if not os.path.exists(p):
        raise AssetError(f"{p} 없음 — {why}. 해당 실험을 먼저 돌릴 것")
P = dict(np.load(PROB))
CNT = {int(k): {int(kk): vv for kk, vv in v.items()} for k, v in json.load(open(CNTJ)).items()}
Y   = np.asarray(P["y_cross"])
REC = np.asarray(P["rec_cross"]).astype(int)
run.log(f"예측 캐시 로드 — {len(Y):,}비트 · 라벨 {len(np.unique(REC))}개"
        f" · N/S/V {np.bincount(Y, minlength=3)[:3].tolist()}")

# ── 매핑 적용 (Q7-B 의 【Q7B-M】 산출물). 없으면 여기서 다시 만든다.
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb
recs = [int(r) for r in wfdb.get_record_list("svdb")]      # ⛔ fallback 없음 (R16)
if len(recs) < 2:
    raise AssetError("SVDB 레코드 목록을 못 받았다 — 연속 번호로 대체하지 않는다")

labels = sorted(int(x) for x in np.unique(REC))
if all(l in recs for l in labels):
    run.log("  라벨이 이미 실제 목록 안에 있다 — 매핑 적용 불필요")
    MAP = {l: l for l in labels}
elif os.path.exists(MAPJ):
    MAP = {int(k): int(v) for k, v in json.load(open(MAPJ)).items()}
    run.log(f"  매핑 로드 — {os.path.basename(MAPJ)} ({len(MAP)}개)")
else:
    CONT = [min(recs) + i for i in range(len(recs))]
    if labels != CONT:
        raise AssetError(f"라벨이 목록 밖인데 연속 가정과도 안 맞는다 — 수동 확인 필요")
    MAP = {l: recs[l - CONT[0]] for l in labels}
    run.log("  매핑 파일이 없어 결정론적으로 재구성했다 (참 = recs[라벨 − 800])")

miss = [l for l in labels if l not in MAP]
if miss:
    raise AssetError(f"매핑에 없는 라벨 {miss[:8]}")
REC = np.array([MAP[int(r)] for r in REC], dtype=np.int64)
assert len(np.unique(REC)) == len(labels), "재라벨에서 레코드가 합쳐졌다 — 매핑이 단사가 아니다"
bad = sorted(set(REC.tolist()) - set(recs))
assert not bad, f"재라벨 후에도 목록 밖 번호가 남았다: {bad[:8]}"

# ── 재검증: (N,S,V) 삼중값이 주석과 맞는가
CNT3 = {r: (v.get(0, 0), v.get(1, 0), v.get(2, 0)) for r, v in CNT.items()}
agree = sum(1 for r in np.unique(REC)
            if CNT3.get(int(r)) and all(abs(x - y) <= 2 for x, y in zip(
                tuple(int((Y[REC == r] == k).sum()) for k in range(3)), CNT3[int(r)])))
run.log(f"  재검증 — (N,S,V) 가 주석과 ±2 안에서 일치: **{agree}/{len(np.unique(REC))}**")
if agree < 0.9 * len(np.unique(REC)):
    raise AssetError("매핑 재검증 실패 — 레코드 번호를 인용할 수 없다")
run.log("  ✅ 매핑 확정")
CONFIG["id_map_recheck"] = {"agree": agree, "n": int(len(np.unique(REC)))}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【Q7B′-A】 **전수 채점** + 반쪽 비교 + 사전등록 관문
from sklearn.metrics import roc_auc_score

def per_record(score, y, rec, recs_, idx, nboot=NB_BOOT, seed=SEED0):
    rng = np.random.RandomState(seed); out = {}
    for r in recs_:
        m = np.where(rec == r)[0]
        if len(m) < 3: continue
        t = (y[m] == idx); s = score[m]
        if not (t.any() and not t.all()): continue
        vals = []
        for _ in range(nboot):
            j = rng.randint(0, len(m), len(m)); tj = t[j]
            if 0 < tj.sum() < len(tj):
                vals.append(roc_auc_score(tj.astype(int), s[j]))
        out[int(r)] = (float(roc_auc_score(t.astype(int), s)),
                       float(np.std(vals, ddof=1)) if len(vals) > 2 else np.nan,
                       int(t.sum()))
    return out

# ── Q7-B 반쪽 실측(ailab-2026-0052) — 비교·순열검정의 기준점. 여기서 재계산하지 않는다.
Q7B_TEST = dict(n=28, macro=0.9256, lo=0.8927, hi=0.9544, width=0.0617,
                se_med=0.0104, se_max=0.0821, dom=0.214)
Q7B_DEV = dict(n=27, macro=0.8382, lo=0.7515, hi=0.9048, width=0.1533,
               se_med=0.0119, se_max=0.0449, dom=0.318)

SC_S = P["v2_cross_raw"].mean(0)[:, IDX_S]      # raw = BN 미적응 (주 arm)
SC_V = P["v2_cross_raw"].mean(0)[:, IDX_V]
ALL = [int(r) for r in np.unique(REC)]

run.log("\n" + "=" * 100)
run.log(f"【Q7B′-A】 전수 채점 · GMIN={GMIN} (Q7-B 에서 승계)")
run.log("=" * 100)

def score_all(score, idx, tag):
    PR = per_record(score, Y, REC, ALL, idx)
    keep = sorted([r for r, (a, se, p) in PR.items() if p >= GMIN and np.isfinite(se)])
    if len(keep) < 2:
        run.log(f"  ⚠️ {tag}: 채점 개체 {len(keep)}개 — 매크로 성립 불가"); return None
    a = np.array([PR[r][0] for r in keep]); se = np.array([PR[r][1] for r in keep])
    pos = np.array([PR[r][2] for r in keep])
    rng = np.random.RandomState(SEED0 + 7)
    bs = [float(a[rng.randint(0, len(a), len(a))].mean()) for _ in range(NB_MACRO)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    drops = np.array([float(np.delete(a, i).mean()) for i in range(len(a))])
    j = int(np.argmax(np.abs(drops - a.mean())))
    infl = np.abs(a - a.mean()) * pos
    return dict(tag=tag, recs=keep, auroc=a.tolist(), se=se.tolist(), pos=pos.tolist(),
                n=len(keep), macro=float(a.mean()), lo=float(lo), hi=float(hi),
                width=float(hi - lo), se_med=float(np.median(se)), se_max=float(se.max()),
                dom=float(pos.max() / max(pos.sum(), 1)),
                drop_rec=int(keep[j]), drop_macro=float(drops[j]),
                burden_rec=int(keep[int(np.argmax(infl))]))

FULL = score_all(SC_S, IDX_S, "전수 · S")
FULL_V = score_all(SC_V, IDX_V, "전수 · V (대조군)")

run.log(f"\n  {'코호트':<20}{'개체':>5}{'매크로':>9}{'레코드 부트 95% CI':>24}{'폭':>8}"
        f"{'중앙SE':>9}{'최대SE':>9}{'지배':>8}")
for D in (FULL, FULL_V):
    if D:
        run.log(f"  {D['tag']:<20}{D['n']:>5}{D['macro']:>9.4f}"
                f"   [{D['lo']:.4f}, {D['hi']:.4f}]{D['width']:>8.4f}"
                f"{D['se_med']:>9.4f}{D['se_max']:>9.4f}{D['dom']:>8.1%}")
for nm, Q in (("(참고) Q7-B TEST", Q7B_TEST), ("(참고) Q7-B DEV", Q7B_DEV)):
    run.log(f"  {nm:<20}{Q['n']:>5}{Q['macro']:>9.4f}   [{Q['lo']:.4f}, {Q['hi']:.4f}]"
            f"{Q['width']:>8.4f}{Q['se_med']:>9.4f}{Q['se_max']:>9.4f}{Q['dom']:>8.1%}")

VERD = {}
def g_(k, ok, d):
    VERD[k] = "✅ 지지" if ok else "❌ 기각"; run.log(f"  {k:<5}{VERD[k]}  {d}")

run.log("")
if FULL is None:
    for k in ("P1", "P2", "P3", "P4", "P5"): VERD[k] = "❌ 기각"
    run.log("  ❌ 전수 매크로가 안 선다 — 전 관문 기각")
    PERM = None
else:
    g_("P1", FULL["n"] >= N_MIN, f"전수 채점 개체 **{FULL['n']}** ≥ {N_MIN}")
    g_("P2", FULL["se_max"] <= SE_MAX,
       f"**최대** 부트 SE **{FULL['se_max']:.4f}** ≤ {SE_MAX} (중앙 {FULL['se_med']:.4f})")
    g_("P3", FULL["dom"] <= DOM_MAX, f"지배 지분 **{FULL['dom']:.1%}** ≤ {DOM_MAX:.0%}")
    g_("P4", FULL["width"] <= CIW_MAX,
       f"매크로 **{FULL['macro']:.4f}** [{FULL['lo']:.4f}, {FULL['hi']:.4f}]"
       f" 폭 **{FULL['width']:.4f}** ≤ {CIW_MAX}")

    # ── P5: 반쪽 격차가 우연인가 — **순열검정**
    #    Q7-B 의 분할은 라벨이 어긋난 채 갈렸으므로 사실상 임의다(ailab-2026-0052 ④).
    #    임의 분할이면 55개체를 무작위로 27/28 로 가른 분포와 비교할 수 있다.
    OBS_GAP = abs(Q7B_TEST["macro"] - Q7B_DEV["macro"])
    av = np.array(FULL["auroc"])
    n1 = Q7B_TEST["n"]
    # ★ 개체 수 확인 없이 n1 을 쓰면 안 된다. 픽스처가 잡았다 — 개체가 6개인데
    #   av[pm[28:]] 가 **빈 배열**이 되어 평균이 nan, 비교가 전부 False, p=0 →
    #   조용히 '기각' 이 나왔다. 나눌 수 없으면 **미결**이다(기각이 아니다).
    if len(av) < n1 + 2 or not np.isfinite(OBS_GAP):
        VERD["P5"] = "⚠️ 미결"
        run.log(f"  P5   ⚠️ 미결  개체 {len(av)}개로는 {n1}/{len(av)-n1} 분할을 못 만든다"
                f" — 순열검정 불가")
        PERM = None
    else:
        rng = np.random.RandomState(SEED0 + 13)
        gaps = np.empty(NB_PERM)
        for i in range(NB_PERM):
            pm = rng.permutation(len(av))
            gaps[i] = abs(av[pm[:n1]].mean() - av[pm[n1:]].mean())
        assert np.isfinite(gaps).all(), "순열 격차에 nan 이 있다 — 분할 크기를 확인할 것"
        pval = float((gaps >= OBS_GAP).mean())
        g_("P5", pval >= 0.05,
           f"반쪽 격차 **{OBS_GAP:.4f}** ({n1} vs {len(av)-n1} 분할) · 순열분포 중앙 "
           f"{np.median(gaps):.4f} · 95분위 {np.percentile(gaps, 95):.4f} → **p = {pval:.4f}**")
        PERM = dict(obs_gap=OBS_GAP, p=pval, median=float(np.median(gaps)),
                    q95=float(np.percentile(gaps, 95)), n_perm=NB_PERM,
                    split=[n1, len(av) - n1])
        # ★ 해석 비대칭을 명시한다.
        #   P5 는 "관측 격차가 **이 코호트의 개체 간 산포**로 설명되나" 를 묻는다.
        #   코호트가 균질할수록 같은 격차가 더 희귀해진다 — 즉 p 는 성능이 아니라
        #   **산포**에 대한 진술이다.
        run.log("       ⚠️ **지지 = 「구조적 차이의 증거가 없다」이지 「차이가 없다」가 아니다.**")
        if pval < 0.05:
            run.log("       ⚠️ 기각이면 두 반쪽이 **교환가능하지 않다**는 뜻이다. 그런데 분할은")
            run.log("          어긋난 라벨로 갈려 사실상 임의였다(ailab-2026-0052 ④) — 임의 분할이")
            run.log("          교환가능하지 않다면 **설명이 필요한 사건**이지 코호트의 결함이 아니다.")
            run.log("          후보: 채점 가능 여부(GMIN·양성/음성 존재)가 분할과 상관됐을 수 있다.")

    run.log(f"\n  ── R11-c: 최대기여 두 정의 ──")
    i1 = FULL["recs"].index(FULL["drop_rec"]); i2 = FULL["recs"].index(FULL["burden_rec"])
    run.log(f"    제거영향 최대  #{FULL['drop_rec']}  AUROC {FULL['auroc'][i1]:.4f}"
            f" · S {FULL['pos'][i1]:,} · 제외 매크로 {FULL['drop_macro']:.4f}"
            f" (Δ {FULL['drop_macro']-FULL['macro']:+.4f})")
    run.log(f"    부담가중 영향  #{FULL['burden_rec']}  AUROC {FULL['auroc'][i2]:.4f}"
            f" · S {FULL['pos'][i2]:,}")
    run.log(f"    ⚠️ 개체 {FULL['n']}개에서 제거영향 관문은 최대 |편차|/{FULL['n']} 만큼만"
            f" 움직인다 — 실측 Δ {abs(FULL['drop_macro']-FULL['macro']):.4f}. 힘이 약하다(R11-c ②)")

    # ── R15: CI 폭 분해
    sv = np.array(FULL["se"]); nn = len(av)
    w_het = 2 * 1.96 * av.std(ddof=1) / np.sqrt(nn)
    w_err = 2 * 1.96 * np.sqrt((sv ** 2).mean()) / np.sqrt(nn)
    run.log(f"\n  ── CI 폭 분해 (R15) ──")
    run.log(f"    관측 {FULL['width']:.4f} · 이질성만 {w_het:.4f} · 측정오차만 {w_err:.4f}"
            f" · 분산비 {av.var(ddof=1)/max((sv**2).mean(),1e-12):.1f}배")
    _dir = "늘렸을" if nn > Q7B_TEST["n"] else "줄였을"
    run.log(f"    개체 {Q7B_TEST['n']} → {nn} 로 {_dir} 때 CI 폭:"
            f" {Q7B_TEST['width']:.4f} → **{FULL['width']:.4f}**"
            f"  (√({Q7B_TEST['n']}/{nn}) 배 예측 "
            f"{Q7B_TEST['width']*np.sqrt(Q7B_TEST['n']/nn):.4f})")

run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
run.log("\n  ⚠️ **사전등록 관문 판정은 Q7-B 의 TEST 값이 최종이다.** 여기 전수 값은")
run.log("     Q2 를 위한 최선 추정으로 쓴다 — 두 역할을 섞지 않는다.")
run.log("  ⚠️ 전이 낙폭 인용 금지(대역 교란 · Q7-C). 실험22-A·§6.5 와 직접 비교 불가.")
CONFIG["result"] = {"verdicts": VERD, "gmin": GMIN, "full_S": FULL, "full_V": FULL_V,
                    "perm": PERM}
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【Q7B′-B】 서술용 부지표 (관문 아님) — PR-AUC · 부담 층화
from sklearn.metrics import average_precision_score
from scipy import stats as _st

F = CONFIG.get("result", {}).get("full_S")
if not F:
    run.log("전수 채점이 없다 — 부지표 생략")
else:
    _V0 = dict(CONFIG["result"]["verdicts"])
    run.log("\n" + "=" * 100)
    run.log("【Q7B′-B】 서술용 부지표 — 관문은 위에서 확정된 채 그대로 둔다")
    run.log("=" * 100)
    au = np.array(F["auroc"]); pos = np.array(F["pos"]); sev = np.array(F["se"])
    pr, prev = [], []
    for r in F["recs"]:
        m = np.where(REC == r)[0]; t = (Y[m] == IDX_S).astype(int)
        pr.append(float(average_precision_score(t, SC_S[m]))); prev.append(float(t.mean()))
    pr = np.array(pr); prev = np.array(prev)
    lift = pr / np.maximum(prev, 1e-9); npr = (pr - prev) / np.maximum(1 - prev, 1e-9)
    rng = np.random.RandomState(SEED0 + 11)
    bs = [float(pr[rng.randint(0, len(pr), len(pr))].mean()) for _ in range(NB_MACRO)]
    plo, phi = np.percentile(bs, [2.5, 97.5])
    run.log(f"\n  전수 매크로 — AUROC {au.mean():.4f} · PR-AUC **{pr.mean():.4f}**"
            f" [{plo:.4f}, {phi:.4f}] · 평균 유병률 {prev.mean():.4f}")
    run.log(f"    (참고) Q7-B TEST 반쪽 PR-AUC 0.4978 [0.3973, 0.6031]")

    run.log(f"\n  ── S 부담별 ── (원 PR-AUC 는 군끼리 비교 금지 — R4)")
    run.log(f"  {'구간':<14}{'개체':>5}{'AUROC':>9}{'PR-AUC':>9}{'유병률':>9}"
            f"{'lift':>8}{'천장':>8}{'정규화PR':>10}{'보유 S':>9}{'지분':>8}")
    for lo_, hi_, tag in ((100, 10**9, "S ≥ 100"), (50, 100, "50 ≤ S < 100"), (0, 50, "S < 50")):
        m = (pos >= lo_) & (pos < hi_)
        if m.sum():
            run.log(f"  {tag:<14}{int(m.sum()):>5}{au[m].mean():>9.4f}{pr[m].mean():>9.4f}"
                    f"{prev[m].mean():>9.4f}{lift[m].mean():>8.1f}"
                    f"{1/max(prev[m].mean(),1e-9):>8.1f}{npr[m].mean():>10.3f}"
                    f"{int(pos[m].sum()):>9,}{pos[m].sum()/pos.sum():>8.1%}")
    rho, prho = _st.spearmanr(pos, au)
    run.log(f"    양성수 vs AUROC 스피어만 rho={rho:+.3f} p={prho:.4f}"
            f"  (Q7-B TEST 반쪽: −0.389 p=0.041)")
    run.log(f"    단순 매크로 {au.mean():.4f} vs 양성수 가중 {np.average(au, weights=pos):.4f}")
    run.log("    ⛔ 가중값으로 갈아타지 않는다(R14-b ② · R11-c ③). 군·개체 비교는 AUROC 로만.")

    run.log(f"\n  ── AUROC 하위 6개체 ──")
    run.log(f"    {'':7}{'AUROC':>8}{'PR-AUC':>9}{'유병률':>9}{'lift':>7}{'천장':>7}{'S':>7}{'SE':>9}")
    for k in np.argsort(au)[:6]:
        run.log(f"    #{F['recs'][k]:<6}{au[k]:>8.4f}{pr[k]:>9.4f}{prev[k]:>9.4f}"
                f"{lift[k]:>7.2f}{1/max(prev[k],1e-9):>7.1f}{pos[k]:>7,}{sev[k]:>9.4f}")
    run.log("    → Q7-D 표적. 심각도 판정은 **AUROC 로** 한다(R15-d).")

    CONFIG["exploratory"] = {"pr_auc_macro": float(pr.mean()),
                             "pr_auc_ci": [float(plo), float(phi)],
                             "prev_mean": float(prev.mean()),
                             "spearman_pos_auroc": [float(rho), float(prho)],
                             "auroc_weighted_by_pos": float(np.average(au, weights=pos)),
                             "per_record": [dict(rec=int(r), auroc=float(a), se=float(s),
                                                 pos=int(p), pr=float(q), prev=float(v))
                                            for r, a, s, p, q, v in
                                            zip(F["recs"], au, sev, pos, pr, prev)]}
    assert CONFIG["result"]["verdicts"] == _V0, "부지표 셀이 관문을 바꿨다 — 사후등록이다"
    run.log("\n  ✅ 관문 판정 불변 확인")
    run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 그림 + 마무리
import matplotlib.pyplot as plt
R = CONFIG.get("result", {}); F = R.get("full_S")
if F:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    o = np.argsort(np.array(F["auroc"]))
    ax[0].errorbar(range(len(o)), np.array(F["auroc"])[o],
                   yerr=1.96 * np.array(F["se"])[o], fmt="o", ms=4, capsize=2)
    ax[0].axhline(F["macro"], color="C1", ls="--", label=f"전수 매크로 {F['macro']:.4f}")
    ax[0].axhspan(F["lo"], F["hi"], color="C1", alpha=.15, label="레코드 부트 95% CI")
    ax[0].axhline(0.9256, color="C2", ls=":", label="Q7-B TEST 0.9256")
    ax[0].axhline(0.8382, color="C3", ls=":", label="Q7-B DEV 0.8382")
    ax[0].set_title(f"① 전수 {F['n']}개체 S AUROC (GMIN={R['gmin']})")
    ax[0].set_xlabel("레코드(AUROC 오름차순)"); ax[0].set_ylabel("AUROC")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if R.get("perm"):
        pm = R["perm"]
        rng = np.random.RandomState(SEED0 + 13); av = np.array(F["auroc"])
        _n1 = pm["split"][0]
        gaps = [abs(av[q[:_n1]].mean() - av[q[_n1:]].mean())
                for q in (rng.permutation(len(av)) for _ in range(4000))]
        ax[1].hist(gaps, bins=50, color="C0", alpha=.7)
        ax[1].axvline(pm["obs_gap"], color="crimson", lw=2,
                      label=f"관측 격차 {pm['obs_gap']:.4f} (p={pm['p']:.3f})")
        ax[1].axvline(pm["q95"], color="gray", ls="--", label=f"95분위 {pm['q95']:.4f}")
        ax[1].set_title("② P5 — 무작위 반쪽 분할의 격차 분포")
        ax[1].set_xlabel("|반쪽 매크로 차|"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); run.save_fig("q7bp_full_rescore", fig); plt.show()

run.finish(R if R else {"verdicts": {}})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-full-rescore`")
